# Comparing methods on a labelled healthcare benchmark

The eICU demo does not include confirmed error labels, so this notebook uses the labelled ADBench annthyroid dataset for a separate comparison. We compare the unsupervised methods, Isolation Forest and an autoencoder, with Gaussian Naive Bayes and a Decision Tree.

The two classifiers learn from the training labels. The unsupervised methods do not use labels when they are fitted. We keep a stratified test set aside for the final evaluation. This benchmark helps us compare methods on annthyroid; it does not show how they would perform on actual health registry errors.

## Load the benchmark data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neural_network import MLPRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_fscore_support,
)

DATA_PATH = Path("data/adbench/2_annthyroid.npz")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download 2_annthyroid.npz from the official ADBench repository and save it to data/adbench/.")
data = np.load(DATA_PATH, allow_pickle=True)
X_raw = pd.DataFrame(data["X"]).apply(pd.to_numeric, errors="coerce")
y = np.asarray(data["y"]).astype(int).ravel()
print("Features:", X_raw.shape, "Anomaly share:", round(float(y.mean()), 4))
display(X_raw.describe().T)
print("Missing values by feature:")
display(X_raw.isna().mean().rename("missing_fraction").to_frame())

Features: (7200, 6) Anomaly share: 0.0742


,count,mean,std,min,25%,50%,75%,max
0,7200.0,0.520518,0.189269,0.0100,0.37000,0.5500,0.6700,0.970
1,7200.0,0.004861,0.022979,0.0000,0.00071,0.0017,0.0027,0.530
2,7200.0,0.019977,0.007539,0.0005,0.01700,0.0201,0.0220,0.180
3,7200.0,0.109430,0.035969,0.0020,0.08900,0.1070,0.1250,0.600
4,7200.0,0.097838,0.019085,0.0170,0.08700,0.0970,0.1040,0.233
5,7200.0,0.113215,0.036098,0.0020,0.09500,0.1110,0.1270,0.642


Missing values by feature:


,missing_fraction
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
5,0.0


The stratified split keeps a similar share of labelled anomalies in the training and test sets. Isolation Forest and the autoencoder are fitted to training features without using labels. Gaussian Naive Bayes and the Decision Tree use training labels. We use five fold stratified cross validation on the training partition to choose the tree depth by average precision. The test labels are left alone until final scoring.

ROC-AUC measures how well a method ranks examples across thresholds. Average precision focuses on finding the less common anomaly class. Precision tells us what share of the flagged rows are labelled anomalies; recall tells us what share of labelled anomalies were found. We also report precision, recall, and F1 for the highest scoring 5% of test rows, to represent a fixed review limit.

In [2]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.30, random_state=42, stratify=y
)

# Fit preprocessing on the training partition only.
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)
scaler = RobustScaler().fit(X_train_imp)
X_train = scaler.transform(X_train_imp)
X_test = scaler.transform(X_test_imp)

# Unsupervised models do not use y_train during fitting.
isolation_forest = IsolationForest(
    n_estimators=200, contamination=0.05, random_state=42
)
isolation_forest.fit(X_train)
if_score = -isolation_forest.decision_function(X_test)

hidden_width = max(2, min(8, X_train.shape[1] // 2))
autoencoder = MLPRegressor(
    hidden_layer_sizes=(hidden_width,), activation="relu", solver="adam",
    max_iter=500, early_stopping=True, random_state=42
)
autoencoder.fit(X_train, X_train)
ae_score = np.mean((X_test - autoencoder.predict(X_test)) ** 2, axis=1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", RobustScaler()),
    ("classifier", GaussianNB()),
])
nb_cv_ap = cross_val_score(
    nb_pipeline, X_train_raw, y_train, cv=cv, scoring="average_precision", n_jobs=1
)
nb_pipeline.fit(X_train_raw, y_train)
nb_score = nb_pipeline.predict_proba(X_test_raw)[:, 1]

tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced", random_state=42, min_samples_leaf=5
    )),
])
tree_search = GridSearchCV(
    tree_pipeline,
    {"classifier__max_depth": [2, 3, 5, 8, None]},
    cv=cv, scoring="average_precision", n_jobs=1, return_train_score=True,
)
tree_search.fit(X_train_raw, y_train)
tree_score = tree_search.predict_proba(X_test_raw)[:, 1]

print("Train/test rows:", len(y_train), len(y_test))
print("Test anomaly share:", round(float(y_test.mean()), 4))
print("GaussianNB mean CV average precision:", round(float(nb_cv_ap.mean()), 4))
print("Decision Tree best CV average precision:", round(float(tree_search.best_score_), 4))
print("Decision Tree selected settings:", tree_search.best_params_)
print("Decision Tree mean train CV score:", round(float(tree_search.cv_results_["mean_train_score"][tree_search.best_index_]), 4))

# MAD baseline is fit on train features and scored on test rows.
train_frame = pd.DataFrame(X_train_imp, columns=X_raw.columns)
test_frame = pd.DataFrame(X_test_imp, columns=X_raw.columns)
center = train_frame.median()
mad = (train_frame - center).abs().median().replace(0, np.nan)
test_z = ((test_frame - center) / (1.4826 * mad)).replace([np.inf, -np.inf], np.nan).fillna(0)
mad_score = test_z.abs().max(axis=1).to_numpy()

score_columns = {
    "MAD baseline": mad_score,
    "Isolation Forest": if_score,
    "Autoencoder": ae_score,
    "Gaussian Naive Bayes": nb_score,
    "Decision Tree": tree_score,
}

def score_summary(name, scores):
    cutoff = np.quantile(scores, 0.95)  # same 5% review capacity for all methods
    predicted = (scores >= cutoff).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predicted, average="binary", zero_division=0
    )
    return {
        "method": name,
        "ROC-AUC": roc_auc_score(y_test, scores),
        "Average precision": average_precision_score(y_test, scores),
        "Precision at 5% flagged": precision,
        "Recall at 5% flagged": recall,
        "F1 at 5% flagged": f1,
    }

metrics = pd.DataFrame([score_summary(name, scores) for name, scores in score_columns.items()])
display(metrics.sort_values("Average precision", ascending=False))

Train/test rows: 5040 2160
Test anomaly share: 0.0741
GaussianNB mean CV average precision: 0.5971
Decision Tree best CV average precision: 0.8841
Decision Tree selected settings: {'classifier__max_depth': None}
Decision Tree mean train CV score: 0.9872


,method,ROC-AUC,Average precision,Precision at 5% flagged,Recall at 5% flagged,F1 at 5% flagged
4,Decision Tree,0.985088,0.868254,0.886957,0.6375,0.741818
0,MAD baseline,0.952794,0.651153,0.685185,0.4625,0.552239
3,Gaussian Naive Bayes,0.907686,0.638845,0.759259,0.5125,0.611940
1,Isolation Forest,0.820288,0.332725,0.407407,0.2750,0.328358
2,Autoencoder,0.451306,0.063013,0.018519,0.0125,0.014925


Gaussian Naive Bayes uses Bayes' rule and assumes the features are conditionally independent given the class. A Decision Tree makes a series of feature based splits; limiting its depth can reduce model complexity. Use the cross validation and final test results to discuss how well the selected tree generalizes and whether there may be overfitting.

Keep the supervised and unsupervised results distinct. The classifiers learned from annthyroid training labels; the anomaly scoring methods did not. This comparison applies to the benchmark only and does not prove that any method can detect errors in eICU or local health records. When discussing precision and recall, remember that anomalies make up a small share of the dataset and that we fixed the review list at 5%.